# 06 - Cleanup and maintenance

Runs as the last task of the pipeline job. Every step logs counts to `ops.pipeline_runs` (task `cleanup`). Set `dry_run = true` to see what would be removed without changing anything.

| Step | Rule (defaults) |
|---|---|
| 1. Landing files | Delete CSVs in the landing volume that bronze has ingested **and** are older than `landing_retention_days` (7). Empty date folders are removed. Files never ingested are never touched |
| 2. Bronze | Delete rows older than `bronze_retention_days` (30), only from bronze runs that silver has processed |
| 3. Rejected | Delete `silver.jobs_rejected` rows older than `rejected_retention_days` (30) |
| 4. Run log | Delete `ops.pipeline_runs` rows older than `run_log_retention_days` (90). Must be longer than bronze retention, because silver uses the run log to know which bronze runs it has processed |
| 5. Expire gold jobs | `is_active = false` for jobs **posted** more than `job_expire_after_days` (2) ago (posted date, else first seen). Scrapers only fetch fresh postings, so "last seen" can't tell if a job is still open |
| 6. Delete gold jobs | Delete expired jobs older than `job_delete_after_days` (2, must be >= expiry) **unless tracked**: kept if any user applied or set a status in `gold.user_job_state` (or the legacy `is_applied` / `application_status` columns say so). Hidden-only jobs are deleted. Jobs `04_enrich` excluded as outside the supported roles (`out_of_scope`) are deleted too, with the same tracking exception. Orphan `gold.job_skills` and `gold.user_job_state` rows are removed |
| 7. Silver | Delete `silver.jobs_clean` / `silver.jobs_enriched` rows older than `job_delete_after_days` that are not in gold (stops the gold merge from re-inserting them) |
| 8. LLM log | Delete failed calls and legacy rows (no `prompt_version`) older than `llm_failed_retention_days` (7), responses from superseded prompt versions older than 30 days (latest kept per endpoint and prompt family: `p*` skills, `r*` roles), and responses for jobs no longer in silver |
| 8b. Embedding cache | Delete failed embeddings and job embeddings whose job is no longer in silver, older than `llm_failed_retention_days`. Role embeddings are kept |
| 9. Delta maintenance | `OPTIMIZE` + `VACUUM` on all pipeline tables, at most every `maintenance_interval_hours` (24) |

Net effect: gold keeps jobs posted in the last 2 days plus every job any user applied to. Older, never-applied jobs (including hidden ones) are removed.

**Job task environment:** no extra dependencies needed.

In [ ]:
%run ./_common

In [ ]:
from datetime import datetime, timedelta, timezone

from pyspark.sql import functions as F

dbutils.widgets.text("landing_path", "/Volumes/jobseeker/default/scrapes", "Landing volume path")
dbutils.widgets.text("landing_retention_days", "7", "Delete ingested landing files after N days")
dbutils.widgets.text("bronze_retention_days", "30", "Delete processed bronze rows after N days")
dbutils.widgets.text("rejected_retention_days", "30", "Delete rejected rows after N days")
dbutils.widgets.text("run_log_retention_days", "90", "Delete pipeline run log rows after N days")
dbutils.widgets.text("llm_failed_retention_days", "7", "Delete failed / legacy LLM log rows after N days")
# Only fresh jobs matter: anything posted more than 2 days ago that you did not apply to is removed
dbutils.widgets.text("job_expire_after_days", "2", "Expire gold jobs posted more than N days ago")
dbutils.widgets.text("job_delete_after_days", "2", "Delete untracked expired jobs posted more than N days ago")
dbutils.widgets.text("maintenance_interval_hours", "24", "Run OPTIMIZE/VACUUM at most every N hours")
dbutils.widgets.dropdown("dry_run", "false", ["false", "true"], "Dry run (report only)")


def positive_int(name):
    value = int(dbutils.widgets.get(name))
    if value < 1:
        raise ValueError(f"{name} must be >= 1, got {value}")
    return value


LANDING_PATH = dbutils.widgets.get("landing_path").rstrip("/")
LANDING_DAYS = positive_int("landing_retention_days")
BRONZE_DAYS = positive_int("bronze_retention_days")
REJECTED_DAYS = positive_int("rejected_retention_days")
RUN_LOG_DAYS = positive_int("run_log_retention_days")
LLM_FAILED_DAYS = positive_int("llm_failed_retention_days")
EXPIRE_DAYS = positive_int("job_expire_after_days")
DELETE_DAYS = positive_int("job_delete_after_days")
MAINTENANCE_HOURS = positive_int("maintenance_interval_hours")
DRY_RUN = dbutils.widgets.get("dry_run") == "true"

if not LANDING_PATH.startswith("/Volumes/"):
    raise ValueError(f"landing_path must be a Unity Catalog volume path: {LANDING_PATH}")
if BRONZE_DAYS <= LANDING_DAYS:
    raise ValueError("bronze_retention_days must be greater than landing_retention_days (landing cleanup checks bronze)")
if RUN_LOG_DAYS <= BRONZE_DAYS:
    raise ValueError("run_log_retention_days must be greater than bronze_retention_days (silver uses the run log)")
if DELETE_DAYS < EXPIRE_DAYS:
    raise ValueError("job_delete_after_days must be >= job_expire_after_days (only expired jobs are deleted)")

BRONZE_TABLE = table("bronze", "jobs_raw")
SILVER_TABLE = table("silver", "jobs_clean")
REJECTED_TABLE = table("silver", "jobs_rejected")
ENRICHED_TABLE = table("silver", "jobs_enriched")
GOLD_TABLE = table("gold", "jobs")
SKILLS_TABLE = table("gold", "job_skills")
USER_STATE_TABLE = table("gold", "user_job_state")
RUNS_TABLE = table("ops", "pipeline_runs")
LLM_LOG_TABLE = table("ops", "enrichment_llm_log")
EMBEDDING_CACHE_TABLE = table("ops", "embedding_cache")
MAINTAINED_TABLES = [BRONZE_TABLE, SILVER_TABLE, REJECTED_TABLE, ENRICHED_TABLE, GOLD_TABLE, SKILLS_TABLE, USER_STATE_TABLE,
                     RUNS_TABLE, LLM_LOG_TABLE, EMBEDDING_CACHE_TABLE]

# Posting age used for expiry: posted date, else the day the job was first scraped
JOB_AGE_DATE = "coalesce(posted_date, CAST(first_seen_at AS DATE))"
# Gold jobs no user tracks: not applied and no status in gold.user_job_state (or the legacy gold.jobs columns)
UNTRACKED_JOB = (
    "NOT coalesce(is_applied, false) AND coalesce(application_status, 'not_applied') = 'not_applied' "
    f"AND NOT EXISTS (SELECT 1 FROM {USER_STATE_TABLE} AS s WHERE s.job_key = t.job_key "
    "AND (coalesce(s.is_applied, false) OR coalesce(s.application_status, 'not_applied') <> 'not_applied'))"
)

# Schema guard: lifecycle columns on gold.jobs (also added by 01_setup)
add_missing_columns(GOLD_TABLE, {"is_active": "BOOLEAN", "expired_at": "TIMESTAMP"})
if not spark.catalog.tableExists(EMBEDDING_CACHE_TABLE):
    raise RuntimeError(f"{EMBEDDING_CACHE_TABLE} does not exist - run 01_setup first")
if not spark.catalog.tableExists(USER_STATE_TABLE):
    raise RuntimeError(f"{USER_STATE_TABLE} does not exist - run 01_setup first")

print(f"dry_run={DRY_RUN} landing={LANDING_DAYS}d bronze={BRONZE_DAYS}d rejected={REJECTED_DAYS}d runs={RUN_LOG_DAYS}d "
      f"llm_failed={LLM_FAILED_DAYS}d expire={EXPIRE_DAYS}d delete={DELETE_DAYS}d maintenance={MAINTENANCE_HOURS}h")

## Helpers

In [ ]:
report = {}

# The target table is aliased `t` in every statement below, so predicates can reference it (see missing_from).


def count_where(table_name, predicate):
    return int(spark.sql(f"SELECT count(*) AS n FROM {table_name} AS t WHERE {predicate}").first()["n"])


def delete_where(step, table_name, predicate):
    """Deletes rows (or only counts them in dry run) and records the count under report[step]."""
    rows = count_where(table_name, predicate)
    if rows and not DRY_RUN:
        spark.sql(f"DELETE FROM {table_name} AS t WHERE {predicate}")
    report[step] = rows
    print(f"{'[DRY RUN] would delete' if DRY_RUN else 'deleted'} {rows:>6} rows  {step}")
    return rows


def update_where(step, table_name, assignments, predicate):
    rows = count_where(table_name, predicate)
    if rows and not DRY_RUN:
        spark.sql(f"UPDATE {table_name} AS t SET {assignments} WHERE {predicate}")
    report[step] = rows
    print(f"{'[DRY RUN] would update' if DRY_RUN else 'updated'} {rows:>6} rows  {step}")
    return rows


def missing_from(table_name, column="job_key"):
    """Rows of the target whose `column` matches no job_key in table_name (orphans).
    NOT EXISTS rather than NOT IN: a plain anti join, and a NULL key in table_name cannot silently block the cleanup."""
    return f"NOT EXISTS (SELECT 1 FROM {table_name} AS ref WHERE ref.job_key = t.{column})"


def sql_literal(value):
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"

## Run cleanup

In [ ]:
with task_run("cleanup") as metrics:
    now = datetime.now(timezone.utc)

    # 1. Landing files already ingested by bronze and older than the retention
    ingested = {
        row["_source_file"].replace("dbfs:", "")
        for row in spark.table(BRONZE_TABLE).select("_source_file").distinct().collect()
        if row["_source_file"]
    }
    cutoff_ms = int((now - timedelta(days=LANDING_DAYS)).timestamp() * 1000)

    def walk(path):
        for info in dbutils.fs.ls(path):
            if info.isDir():
                yield from walk(info.path)
            else:
                yield info

    removed_files, kept_not_ingested = [], 0
    for info in walk(LANDING_PATH):
        file_path = info.path.replace("dbfs:", "")
        if file_path not in ingested:
            kept_not_ingested += 1
            continue
        if info.modificationTime < cutoff_ms:
            if not DRY_RUN:
                dbutils.fs.rm(info.path)
            removed_files.append(file_path)

    removed_folders = 0
    if not DRY_RUN:
        for folder in dbutils.fs.ls(LANDING_PATH):
            if folder.isDir() and not dbutils.fs.ls(folder.path):
                dbutils.fs.rm(folder.path, recurse=True)
                removed_folders += 1
    report["landing_files_deleted"] = len(removed_files)
    report["landing_folders_deleted"] = removed_folders
    report["landing_files_not_ingested"] = kept_not_ingested
    print(f"{'[DRY RUN] would delete' if DRY_RUN else 'deleted'} {len(removed_files):>6} landing files "
          f"({removed_folders} empty folders, {kept_not_ingested} not yet ingested kept)")

    # 2. Bronze rows from runs silver has processed
    spark.sql(f"""
        SELECT DISTINCT explode(from_json(details, 'struct<bronze_run_ids:array<string>>').bronze_run_ids) AS run_id
        FROM {RUNS_TABLE}
        WHERE task = 'silver_transform' AND status = 'SUCCESS'
    """).createOrReplaceTempView("processed_bronze_runs")
    delete_where(
        "bronze_rows", BRONZE_TABLE,
        f"_ingested_at < current_timestamp() - INTERVAL {BRONZE_DAYS} DAYS "
        f"AND _job_run_id IN (SELECT run_id FROM processed_bronze_runs)",
    )

    # 3. Rejected rows and 4. run log
    delete_where("rejected_rows", REJECTED_TABLE, f"_rejected_at < current_timestamp() - INTERVAL {REJECTED_DAYS} DAYS")
    delete_where("run_log_rows", RUNS_TABLE, f"started_at < current_timestamp() - INTERVAL {RUN_LOG_DAYS} DAYS")

    # 5. Expire / re-activate gold jobs by posting age
    update_where(
        "gold_expired", GOLD_TABLE, "is_active = false, expired_at = current_timestamp()",
        f"coalesce(is_active, true) AND {JOB_AGE_DATE} < date_sub(current_date(), {EXPIRE_DAYS})",
    )
    update_where(
        "gold_reactivated", GOLD_TABLE, "is_active = true, expired_at = NULL",
        f"(is_active IS NULL OR NOT is_active) AND {JOB_AGE_DATE} >= date_sub(current_date(), {EXPIRE_DAYS})",
    )

    # 6. Delete old expired jobs nobody tracks (applied / saved / interviewing ... by any user are kept forever)
    delete_where(
        "gold_deleted", GOLD_TABLE,
        f"NOT coalesce(is_active, true) AND {JOB_AGE_DATE} < date_sub(current_date(), {DELETE_DAYS}) AND {UNTRACKED_JOB}",
    )
    # 6b. Jobs 04_enrich excluded as outside the supported roles (kept if any user tracks them)
    delete_where(
        "gold_out_of_scope", GOLD_TABLE,
        f"EXISTS (SELECT 1 FROM {ENRICHED_TABLE} AS e WHERE e.job_key = t.job_key AND NOT e.gold_eligible "
        f"AND e.gold_exclusion_reason LIKE '%out_of_scope%') AND {UNTRACKED_JOB}",
    )
    delete_where("gold_skills_orphans", SKILLS_TABLE, missing_from(GOLD_TABLE))
    delete_where("user_state_orphans", USER_STATE_TABLE, missing_from(GOLD_TABLE))

    # 7. Old silver rows that are not in gold (incl. non-India / non-English jobs)
    delete_where(
        "silver_rows", SILVER_TABLE,
        f"{JOB_AGE_DATE} < date_sub(current_date(), {DELETE_DAYS}) AND {missing_from(GOLD_TABLE)}",
    )
    delete_where("enriched_orphans", ENRICHED_TABLE, missing_from(SILVER_TABLE))

    # 8. LLM log: failures, legacy rows, superseded prompt versions, orphans.
    #    Prompt family = first letter of prompt_version (p = skills, r = roles); the latest version per endpoint + family is kept.
    delete_where(
        "llm_failed_or_legacy", LLM_LOG_TABLE,
        f"(error IS NOT NULL OR response IS NULL OR prompt_version IS NULL) "
        f"AND called_at < current_timestamp() - INTERVAL {LLM_FAILED_DAYS} DAYS",
    )
    current_prompts = (
        spark.table(LLM_LOG_TABLE).where("prompt_version IS NOT NULL")
        .groupBy("endpoint", F.substring("prompt_version", 1, 1).alias("family"))
        .agg(F.max_by("prompt_version", "called_at").alias("prompt_version"))
        .collect()
    )
    if current_prompts:
        keep = " OR ".join(
            f"(endpoint = {sql_literal(row['endpoint'])} AND prompt_version = {sql_literal(row['prompt_version'])})"
            for row in current_prompts
        )
        delete_where(
            "llm_old_prompt_versions", LLM_LOG_TABLE,
            f"prompt_version IS NOT NULL AND NOT ({keep}) AND called_at < current_timestamp() - INTERVAL 30 DAYS",
        )
    delete_where(
        "llm_orphans", LLM_LOG_TABLE,
        f"{missing_from(SILVER_TABLE)} AND called_at < current_timestamp() - INTERVAL {LLM_FAILED_DAYS} DAYS",
    )

    # 8b. Embedding cache: failed calls and embeddings of jobs no longer in silver. Role embeddings ('role:' keys) are kept.
    delete_where(
        "embedding_failed", EMBEDDING_CACHE_TABLE,
        f"embedding IS NULL AND created_at < current_timestamp() - INTERVAL {LLM_FAILED_DAYS} DAYS",
    )
    delete_where(
        "embedding_orphans", EMBEDDING_CACHE_TABLE,
        f"cache_key NOT LIKE 'role:%' AND {missing_from(SILVER_TABLE, column='cache_key')} "
        f"AND created_at < current_timestamp() - INTERVAL {LLM_FAILED_DAYS} DAYS",
    )

    # 9. Delta maintenance, at most every MAINTENANCE_HOURS
    last_maintenance = spark.sql(f"""
        SELECT max(started_at) AS last_run FROM {RUNS_TABLE}
        WHERE task = 'cleanup' AND status = 'SUCCESS'
          AND from_json(details, 'struct<maintenance_ran:boolean>').maintenance_ran
    """).first()["last_run"]
    maintenance_due = last_maintenance is None or (
        now - last_maintenance.replace(tzinfo=timezone.utc) >= timedelta(hours=MAINTENANCE_HOURS)
    )
    maintenance_errors = {}
    if maintenance_due and not DRY_RUN:
        for table_name in MAINTAINED_TABLES:
            try:
                spark.sql(f"OPTIMIZE {table_name}")
                spark.sql(f"VACUUM {table_name}")
                print(f"maintained {table_name}")
            except Exception as exc:  # maintenance must never fail the pipeline
                maintenance_errors[table_name] = str(exc).splitlines()[0][:300]
                print(f"[WARN] maintenance failed for {table_name}: {maintenance_errors[table_name]}")
    else:
        print(f"maintenance skipped (due={maintenance_due}, last={last_maintenance}, dry_run={DRY_RUN})")

    total_removed = sum(value for key, value in report.items() if key not in ("landing_files_not_ingested", "gold_reactivated"))
    metrics.update(rows_in=None, rows_out=total_removed, rows_rejected=0, files_read=len(removed_files),
                   message="dry run" if DRY_RUN else None)
    metrics["details"].update(
        report, dry_run=DRY_RUN, maintenance_ran=bool(maintenance_due and not DRY_RUN),
        maintenance_errors=maintenance_errors,
    )

In [ ]:
stop_if_job()  # scheduled runs end here; the cells below are for manual inspection

## Inspect

In [ ]:
display(spark.createDataFrame([(step, int(value)) for step, value in report.items()], "step STRING, rows INT"))

In [ ]:
display(
    spark.table(GOLD_TABLE)
    .groupBy(F.coalesce("is_active", F.lit(True)).alias("is_active"), "application_status")
    .count()
    .orderBy("is_active", "application_status")
)

In [ ]:
display(spark.table(RUNS_TABLE).where("task = 'cleanup'").orderBy(F.col("started_at").desc()).limit(5))